# GRPO Training: RTS Commander Agent

Train a LoRA adapter on **LFM2.5-1.2B-Instruct** using TRL's `GRPOTrainer` with the RTS game engine as environment.

**Hardware**: Colab T4 (free tier) — 16GB VRAM, sufficient for 1.17B params.
**Base model**: `LiquidAI/LFM2.5-1.2B-Instruct` (~900MB, no quantization needed)

The agent plays Red Alert 2 via text commands (`build`, `produce`, `move`, `attack`, `stop`, `guard`) against a scripted SimpleAI opponent.

## 1. Mount Google Drive

Adapter checkpoints and logs will be saved to Google Drive so they persist across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/rts_commander'
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_DIR}')

## 2. Install Dependencies

In [ ]:
# Core deps — TRL requires transformers >= 5.2.0 for environment_factory
!pip install transformers>=5.2.0 trl peft accelerate bitsandbytes datasets 2>&1 | tail -5

# Verify transformers version
import transformers
print(f'transformers {transformers.__version__}')
assert transformers.__version__ >= '5.2.0', (
    f'Need transformers>=5.2.0, got {transformers.__version__}. '
    'Run: !pip install git+https://github.com/huggingface/transformers'
)

## 3. Install RTS Engine

Clone the repo and install `rts_engine` as a package. The engine is pure Python with zero external dependencies.

In [ ]:
# Clone the repo (sparse checkout — only colab_dev)
!git clone --depth 1 https://github.com/slothitude/fractal-graph.git /tmp/fg 2>&1 | tail -3
%cd /tmp/fg/colab_dev
!pip install -e . 2>&1 | tail -3

# Quick smoke test
from rts_engine.engine import GameEngine
from rts_engine.state_encoder import encode_state
from rts_engine.action_vocab import format_tools_prompt
from rts_engine.opponents.simple_ai import SimpleAI
from rts_engine.rewards import combined_reward
print(f'encode_state: {encode_state(GameEngine(seed=0).get_state(0))[:60]}...')
print(f'tools prompt: {format_tools_prompt()[:60]}...')
print('Engine installed OK')

## 4. TRL-Compatible Environment Wrapper

TRL's `environment_factory` auto-discovers public methods as tool calls.
Each method becomes a tool the model can invoke during generation.

In [ ]:
from rts_engine.engine import GameEngine
from rts_engine.state_encoder import encode_state
from rts_engine.action_vocab import format_tools_prompt, parse_action_text
from rts_engine.opponents.simple_ai import SimpleAI
from rts_engine.rewards import combined_reward


class CommanderToolEnv:
    """TRL environment_factory compatible wrapper.

    Public methods (build, produce, move, attack, stop, guard) are
    auto-exposed as tools by TRL's GRPOTrainer.

    The model plays faction 0 (bottom-left), SimpleAI plays faction 1 (top-right).
    """

    TICKS_PER_ACTION = 5  # engine ticks simulated between model turns

    def reset(self, **kwargs) -> str:
        """Start a new game. Returns initial observation text."""
        seed = kwargs.get('seed', 42)
        self.engine = GameEngine(map_size=16, max_ticks=300, seed=seed)
        self.engine.setup_two_player()
        self.opponent = SimpleAI(1)
        self._reward = 0.0
        self._prev_reward = 0.0
        self._action_count = 0
        return self._observe()

    def build(self, structure_type: str, x: int, y: int) -> str:
        """Build a structure at a position.

        Args:
            structure_type: barracks or war_factory
            x: grid x position
            y: grid y position
        """
        self.engine.execute_action(0, {
            'action': 'build', 'structure_type': structure_type, 'x': x, 'y': y
        })
        return self._step_and_observe()

    def produce(self, unit_type: str, structure_id: int) -> str:
        """Queue a unit for production.

        Args:
            unit_type: gi, grizzly, or prism
            structure_id: the structure to produce from
        """
        self.engine.execute_action(0, {
            'action': 'produce', 'unit_type': unit_type, 'structure_id': structure_id
        })
        return self._step_and_observe()

    def move(self, unit_ids: str, x: int, y: int) -> str:
        """Move units to a target position.

        Args:
            unit_ids: comma-separated unit IDs (e.g. "1,2,3")
            x: target x
            y: target y
        """
        uids = [int(i.strip()) for i in unit_ids.split(',')]
        self.engine.execute_action(0, {'action': 'move', 'unit_ids': uids, 'x': x, 'y': y})
        return self._step_and_observe()

    def attack(self, unit_ids: str, target_id: int) -> str:
        """Order units to attack a target.

        Args:
            unit_ids: comma-separated attacking unit IDs
            target_id: enemy unit/structure to attack
        """
        uids = [int(i.strip()) for i in unit_ids.split(',')]
        self.engine.execute_action(0, {'action': 'attack', 'unit_ids': uids, 'target_id': target_id})
        return self._step_and_observe()

    def stop(self, unit_ids: str) -> str:
        """Cancel current orders for units.

        Args:
            unit_ids: comma-separated unit IDs to stop
        """
        uids = [int(i.strip()) for i in unit_ids.split(',')]
        self.engine.execute_action(0, {'action': 'stop', 'unit_ids': uids})
        return self._step_and_observe()

    def guard(self, unit_ids: str, x: int, y: int) -> str:
        """Move units to patrol/defend a position.

        Args:
            unit_ids: comma-separated unit IDs
            x: patrol center x
            y: patrol center y
        """
        uids = [int(i.strip()) for i in unit_ids.split(',')]
        self.engine.execute_action(0, {'action': 'guard', 'unit_ids': uids, 'x': x, 'y': y})
        return self._step_and_observe()

    # --- Internal methods (not exposed as tools) ---

    def _step_and_observe(self) -> str:
        """Simulate engine ticks with opponent acting, then return observation."""
        self._action_count += 1
        for _ in range(self.TICKS_PER_ACTION):
            if self.engine.game_over:
                break
            opp = self.opponent.get_action(self.engine)
            if opp:
                self.engine.execute_action(1, opp)
            self.engine.tick()
        self._update_reward()
        return self._observe()

    def _observe(self) -> str:
        state = self.engine.get_state(0)
        return encode_state(state) + '\n' + format_tools_prompt()

    def _update_reward(self):
        fs = self.engine.factions[0]
        efs = self.engine.factions[1]
        my_units = sum(1 for u in fs.units.values() if u.is_alive)
        enemy_units = sum(1 for u in efs.units.values() if u.is_alive)
        my_hp = sum(s.hp for s in fs.structures.values() if s.is_alive)
        enemy_hp = sum(s.hp for s in efs.structures.values() if s.is_alive)
        result = None
        if self.engine.game_over:
            result = 'win' if self.engine.winner == 0 else ('loss' if self.engine.winner == 1 else 'draw')
        self._prev_reward = self._reward
        self._reward += combined_reward(result, my_units, enemy_units, my_hp, enemy_hp)


# Test the environment
env = CommanderToolEnv()
obs = env.reset(seed=42)
print(f'Initial obs ({len(obs)} chars): {obs[:120]}...')
obs2 = env.build('barracks', 3, 3)
print(f'After build ({len(obs2)} chars): {obs2[:120]}...')
print(f'Reward so far: {env._reward:.4f}, Actions: {env._action_count}')

## 5. Reward Function

TRL calls this after each generation. We return the cumulative episode reward.

In [ ]:
def reward_func(environments, **kwargs):
    """Return reward for each environment instance."""
    return [env._reward for env in environments]

# Quick sanity check
test_env = CommanderToolEnv()
test_env.reset(seed=99)
for _ in range(10):
    if test_env.engine.game_over:
        break
    test_env.build('barracks', 3, 3)
print(f'Reward after 10 builds: {test_env._reward:.4f}')
print(reward_func([test_env]))

## 6. Training Dataset

Each prompt is a system message instructing the model to be an RTS commander.
Multiple seeds create diverse game scenarios.

In [ ]:
from datasets import Dataset

SYSTEM_MSG = (
    "You are a Red Alert 2 commander. Use the available tools to build structures, "
    "produce units, move armies, and attack the enemy base. "
    "Win by destroying all enemy structures. You start at the bottom-left corner. "
    "Build barracks to produce infantry (gi), then war_factory for tanks (grizzly, prism). "
    "Balance economy (credits) with military production. Attack when you have an advantage."
)

NUM_PROMPTS = 500
prompts = [{"role": "user", "content": SYSTEM_MSG}] * NUM_PROMPTS
dataset = Dataset.from_dict({"prompt": [prompts]})

print(f'Dataset: {len(dataset)} prompts, sample: {dataset[0]["prompt"][0]["content"][:80]}...')

## 7. GRPO Training

Uses LoRA (r=8) to keep trainable params low. The `environment_factory` tells
TRL to use `CommanderToolEnv` — its public methods become tools the model can call.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer
from peft import LoraConfig, TaskType

MODEL_ID = "LiquidAI/LFM2.5-1.2B-Instruct"
OUTPUT_DIR = f'{DRIVE_DIR}/commander_adapter_v1'

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

training_args = GRPOConfig(
    # Schedule
    learning_rate=1e-5,
    warmup_steps=25,
    max_steps=500,

    # Batching — T4 is 16GB, 1.17B model fits comfortably
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,

    # Generation
    max_completion_length=128,

    # Optimization
    optim="paged_adamw_8bit",
    fp16=True,

    # Saving — checkpoints go to Google Drive
    output_dir=OUTPUT_DIR,
    logging_steps=25,
    save_steps=250,
    report_to="none",

    # Disable thinking/reasoning tokens for this model
    chat_template_kwargs={"enable_thinking": False},
)

trainer = GRPOTrainer(
    model=MODEL_ID,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=reward_func,
    environment_factory=CommanderToolEnv,
    peft_config=peft_config,
)

# Show trainable params
trainer.model.print_trainable_parameters()
print(f'Output dir: {OUTPUT_DIR}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')

In [ ]:
# Train! This will take ~30-60 min on a free T4.
trainer.train()

## 8. Save Adapter to Google Drive

In [ ]:
trainer.save_model(OUTPUT_DIR)
trainer.processing_class.save_pretrained(OUTPUT_DIR)

# List saved files
import os
files = sorted(os.listdir(OUTPUT_DIR))
print(f'Saved {len(files)} files to {OUTPUT_DIR}:')
for f in files:
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f'  {f} ({size:,} bytes)')

## 9. Evaluate: Trained Adapter vs SimpleAI

Run 20 games and compare win rate against the baseline SimpleAI.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import re
import random

# Load trained adapter
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto', torch_dtype=torch.float16)
model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
model.eval()


def get_model_action(obs_text: str) -> str:
    """Generate an action from the model given current observation."""
    messages = [{"role": "user", "content": SYSTEM_MSG + '\n\n' + obs_text}]
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
    with torch.no_grad():
        output = model.generate(inputs, max_new_tokens=64, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    generated = tokenizer.decode(output[0][inputs.shape[1]:], skip_special_tokens=True).strip()
    return generated


def extract_action(text: str) -> str | None:
    """Extract a valid action command from model output."""
    for line in text.split('\n'):
        line = line.strip()
        action = parse_action_text(line)
        if action:
            return line
    return None


def run_eval_game(seed: int) -> dict:
    """Run one evaluation game, return stats."""
    env = CommanderToolEnv()
    obs = env.reset(seed=seed)
    actions_taken = []
    build_actions = set()

    for step in range(60):  # max 60 model turns
        if env.engine.game_over:
            break
        raw = get_model_action(obs)
        action_text = extract_action(raw)
        if action_text:
            actions_taken.append(action_text)
            if action_text.startswith('build'):
                build_actions.add(action_text.split()[1])
            # Use the step() method from the existing CommanderEnv for evaluation
            from rts_engine.adapters.commander_env import CommanderEnv
            # Actually, use the tool env directly
            action = parse_action_text(action_text)
            if action:
                env.engine.execute_action(0, action)
                for _ in range(env.TICKS_PER_ACTION):
                    if env.engine.game_over:
                        break
                    opp = env.opponent.get_action(env.engine)
                    if opp:
                        env.engine.execute_action(1, opp)
                    env.engine.tick()
                env._update_reward()
                obs = env._observe()

    return {
        'winner': env.engine.winner,
        'ticks': env.engine.tick_count,
        'reward': env._reward,
        'actions': len(actions_taken),
        'build_diversity': len(build_actions),
    }


# Run evaluation
NUM_EVAL = 20
results = [run_eval_game(seed=i) for i in range(NUM_EVAL)]

wins = sum(1 for r in results if r['winner'] == 0)
losses = sum(1 for r in results if r['winner'] == 1)
draws = NUM_EVAL - wins - losses
avg_ticks = sum(r['ticks'] for r in results) / NUM_EVAL
avg_actions = sum(r['actions'] for r in results) / NUM_EVAL
avg_reward = sum(r['reward'] for r in results) / NUM_EVAL
avg_build_div = sum(r['build_diversity'] for r in results) / NUM_EVAL

print('=' * 50)
print(f'EVALUATION: {NUM_EVAL} games vs SimpleAI')
print(f'  Win:  {wins} ({wins/NUM_EVAL*100:.0f}%)')
print(f'  Loss: {losses} ({losses/NUM_EVAL*100:.0f}%)')
print(f'  Draw: {draws} ({draws/NUM_EVAL*100:.0f}%)')
print(f'  Avg ticks: {avg_ticks:.0f}')
print(f'  Avg actions/episode: {avg_actions:.1f}')
print(f'  Avg reward: {avg_reward:.3f}')
print(f'  Avg build diversity: {avg_build_div:.1f}')
print('=' * 50)

## 10. Fallback: Custom Training Loop

If TRL's `environment_factory` doesn't work (e.g. `transformers<5.2.0`),
this cell provides a manual GRPO rollout loop using the gym-style `CommanderEnv.step()`.

In [ ]:
# Fallback: custom GRPO loop if TRL environment_factory fails.
# Uncomment and run this cell ONLY if the main training cell errors.

'''
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from rts_engine.adapters.commander_env import CommanderEnv

MODEL_ID = "LiquidAI/LFM2.5-1.2B-Instruct"

# Load model + LoRA
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, padding_side='left')
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map='auto', torch_dtype=torch.float16)
peft_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, peft_config)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

SYSTEM_MSG = "You are a Red Alert 2 commander. Issue commands to build, produce, move, and attack."

def generate_action(model, tokenizer, obs_text):
    prompt = SYSTEM_MSG + "\\n" + obs_text + "\\nAction:"
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=32, temperature=0.7, do_sample=True, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

NUM_EPISODES = 200
NUM_GENERATIONS = 4  # for GRPO advantage estimation

for episode in range(NUM_EPISODES):
    env = CommanderEnv(map_size=16, seed=episode)
    obs = env.reset()
    
    # Generate multiple trajectories for GRPO
    trajectories = []
    for g in range(NUM_GENERATIONS):
        env_g = CommanderEnv(map_size=16, seed=episode)
        obs_g = env_g.reset()
        episode_reward = 0.0
        done = False
        while not done:
            action_text = generate_action(model, tokenizer, obs_g)
            obs_g, reward, done, info = env_g.step(action_text)
            episode_reward += reward
        trajectories.append((episode_reward, obs_g))
    
    # GRPO advantage: reward - mean(rewards)
    rewards = [t[0] for t in trajectories]
    mean_r = sum(rewards) / len(rewards)
    advantages = [r - mean_r for r in rewards]
    
    # Policy gradient update on best trajectory
    best_idx = max(range(NUM_GENERATIONS), key=lambda i: rewards[i])
    if advantages[best_idx] > 0:  # only update if above baseline
        loss = -advantages[best_idx] * torch.tensor(1.0)  # simplified
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        optimizer.zero_grad()
    
    if episode % 25 == 0:
        print(f'Episode {episode}: mean_reward={mean_r:.3f}, best={max(rewards):.3f}')

model.save_pretrained(f'{DRIVE_DIR}/commander_adapter_v1')
tokenizer.save_pretrained(f'{DRIVE_DIR}/commander_adapter_v1')
print('Fallback training complete.')
'''

## 11. Push Adapter to HuggingFace Hub (optional)

If you want to share the trained adapter publicly.

In [ ]:
# Uncomment to push to HuggingFace Hub
# !pip install huggingface_hub
# from huggingface_hub import create_repo, HfApi
#
# REPO_ID = "slothitude/rts-commander-lfm2.5-v1"  # change to your username
# create_repo(repo_id=REPO_ID, exist_ok=True, private=False)
#
# api = HfApi()
# api.upload_folder(folder_path=OUTPUT_DIR, repo_id=REPO_ID, repo_type="model")
# print(f'Pushed to https://huggingface.co/{REPO_ID}')